### 获取菌种在NCBI数据库的高通量数据
- Myceliophthora thermophila可换成其他菌种
- key_name = 'Mt', 主要是保存表格名称开头
- 先安装以下需要的包

In [ ]:
%pip install biopython
%pip install pandas
%pip install openpyxl

In [1]:
from Bio import Entrez
import pandas as pd
import re


# 指定Entrez使用的邮件地址，这是必需的
Entrez.email = "3575318018@qq.com"

# 保存表格的名称开头，便于分类
key_name = 'Mt'

# 指定搜索的关键词和数据库
search_term = "Myceliophthora thermophila AND GSE[Entry Type]"
database = "gds"

# 使用Entrez进行搜索并获取ID列表
handle1 = Entrez.esearch(db=database, term=search_term, retstart=0, retmax=200)
record = Entrez.read(handle1)
id_list = record["IdList"]

# 获取所有ID的详细信息
handle2 = Entrez.esummary(db=database, id=",".join(id_list))
records = Entrez.read(handle2)
records = list(records)


def get_gse_info(records):
    """根据records,获取相关菌种的gse数据集关键信息.
    """
    gse_list = []
    for record in records:
        title = record["title"]
        summary = record["summary"]
        Gse_id = record['Accession']
        samples_num = record['n_samples']
        
        if "PubMedIds" in record and len(record["PubMedIds"]) > 0:
            pmid = str(record["PubMedIds"][0])
            match = re.search(r'\d+', pmid)
            if match:
                pmid = match.group(0)
            else:
                pmid = "N/A"
        else:
            pmid = "N/A"

        gse_list.append({'GSE Series':Gse_id,'Title':title,'Summary':summary,'Samples':samples_num,'PMID':pmid})

    df_gse = pd.DataFrame(gse_list)
    df_gse.to_excel(key_name + '_' +'gse_info_raw.xlsx', index=False, sheet_name='Sheet1', columns=['GSE Series',
                                                                                                    'Title',
                                                                                                    'Summary',
                                                                                                    'Samples',
                                                                                                    'PMID'])
    
    # 将df_gse中GSE Series的GSE号保存在一个txt文件中，行尾序列为LF格式
    with open(key_name + '_' + 'gse.txt', 'w') as f:
        for i in df_gse['GSE Series']:
            f.write(i + '\n')

    return df_gse

def get_gsm_and_sample_name(records):
    """根据records,获取 GSM Sample ID 与 name 及样本对应的GSE Series ID.
    """
    sample_list = []
    for record in records:
        Gse_id = record['Accession']
        samples = record['Samples']
        for key in samples:
            accession = key['Accession']
            title = key['Title']
            sample_list.append({'GEO Series': Gse_id,'Accession': accession,'Title': title})
            
    df = pd.DataFrame(sample_list)
    df.to_excel(key_name + '_' + 'gsm_id.xlsx', index=False, sheet_name='Sheet1', columns=['GEO Series','Accession','Title'])
    
    return df

In [2]:
get_gse_info(records)

,GSE Series,Title,Summary,Samples,PMID
0,GSE213576,LaeA regulates mycelium growth by mediating H3...,This SuperSeries is composed of the SubSeries ...,36,37013645
1,GSE213575,LaeA regulates mycelium growth by mediating H3...,The low efficiency of fungal growth and substr...,4,37013645
2,GSE213573,LaeA regulates mycelium growth by mediating H3...,The low efficiency of fungal growth and substr...,8,37013645
3,GSE213570,LaeA regulates mycelium growth by mediating H3...,The low efficiency of fungal growth and substr...,24,37013645
4,GSE221464,The arabinose transporter MtLat-1 is involved ...,"In this study, we investigated the role of L-a...",24,36966330
5,GSE214002,Weimberg pathway an alternative for utilizatio...,"An alternative for utilization of D-xylose, th...",8,36691040
6,GSE214142,PFK2/FBPase-2 is a potential target for metabo...,The PFK-2/ FBPase-2 are important regulatory e...,18,36478865
7,GSE205182,"MtTRC-1, a Novel Transcription Factor, Regulat...","In thermophilic fungi, work on the ER stress a...",18,36165620
8,GSE184074,Reconstruction and analysis genome-scale metab...,Myceliophthora thermophila is a thermophilic f...,12,35257374
9,GSE183387,Transcriptomic insights into the roles of the ...,"Thermothelomyces thermophilus, formerly known ...",49,N/A


In [3]:
get_gsm_and_sample_name(records)

,GEO Series,Accession,Title
0,GSE213576,GSM6589279,Δcre-1-Glu_2
1,GSE213576,GSM6589282,ΔlaeA-Avi-2d_1
2,GSE213576,GSM6589270,ΔlaeA-Glu_1
3,GSE213576,GSM6589316,ΔlaeA_H3K9me3_Input-2
4,GSE213576,GSM6589310,WT_H3K9me3_ChIP-2
...,...,...,...
277,GSE27323,GSM675519,Myceliophthora_thermophila_Barley_34C_rep1
278,GSE27323,GSM675532,Thielavia_terrestris_Glucose_34C_rep2
279,GSE27323,GSM675521,Myceliophthora_thermophila_Glucose_45C_rep1
280,GSE27323,GSM675535,Chaetomium_globosum_Barley_34C_rep1


In [ ]:
# for record in records:
#     Samples_name = record['Samples']
#    #  for i in Samples_name:
#       #  Samples_name_singel = Samples_name['Accession']
#       #  Samples_name_title = Samples_name['Title']

#     print("Samples_name: {}".format(Samples_name))

    # print("Title: {}".format(title))
    # print("Summary: {}".format(summary))
    # print("Pmid: {}".format(pmid))
    # print("GSE: {}".format(Gse_id))
    # print("Samples: {}".format(samples_num))

# for record in records:
#     samples = record['Samples']
#     for sample in samples:
#         accession = sample['Accession']
#         title = sample['Title']
#         print("Accession: {}, Title: {}".format(accession, title))